# Photo Distraction Detection

Classifies a single uploaded photo instead of a live video source. If it's confidently a
distraction class, sends a real alert to the backend/mobile app and sounds the physical
buzzer, same as the live-camera/HDMI/video-file notebooks.

**Why the photo gets "replayed" 8 times through the alert logic:** the alert only fires
after 8 *consecutive* confident detections (see `alert_loop_infer.py`'s `AlertHysteresis`)
- that's what stops one noisy video frame from causing a false alarm. A single photo is
only one data point, so rather than inventing a separate, looser rule just for photos,
this notebook feeds the photo's classification result through that exact same hysteresis
logic 8 times: "if this photo were what the camera saw for 8 sustained ticks, would the
real system fire?" That keeps the verdict consistent with what the live notebooks would
actually decide for the same content, not a different standard.

**How to use:** upload a photo via Jupyter's file browser (the Upload button on the file
tree view), then set `IMAGE_PATH` below to point at it.

In [ ]:
import numpy as np
import cv2
from tflite_runtime import interpreter as tflite
import matplotlib.pyplot as plt
%matplotlib inline

from alert_loop_infer import AlertHysteresis, LABEL_NAMES, CONFIDENCE_THRESHOLD, \
    post_alert, sound_buzzer_alert

MODEL_PATH = "/home/xilinx/mobilenetv2_crossview_finetuned_int8.tflite"
IMAGE_PATH = "/home/xilinx/test_photo.jpg"  # change to your uploaded photo's path

interp = tflite.Interpreter(model_path=MODEL_PATH)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print("Model loaded:", MODEL_PATH)

In [ ]:
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    raise RuntimeError(f"Could not read {IMAGE_PATH} - check the path and that the file was uploaded")

img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
resized = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)
x = resized.astype(inp['dtype'])[None, ...]

interp.set_tensor(inp['index'], x)
interp.invoke()
y = interp.get_tensor(out['index'])[0]

class_id = int(np.argmax(y))
confidence = float(y[class_id])
label = LABEL_NAMES[class_id]

plt.figure(figsize=(6, 4.5))
plt.imshow(img)
plt.title(f"{label} ({confidence:.0%})", fontsize=12)
plt.axis("off")
plt.show()

print(f"Predicted class: {label}")
print(f"Confidence: {confidence:.1%}")
print(f"Meets confidence threshold ({CONFIDENCE_THRESHOLD:.0%})? {'yes' if confidence >= CONFIDENCE_THRESHOLD else 'no'}")

In [ ]:
hysteresis = AlertHysteresis()
fired = False
for tick in range(8):
    fired = hysteresis.tick(class_id, confidence)

if fired:
    print(f"*** DISTRACTION DETECTED: {label} ({confidence:.0%}) - alerting ***")
    sound_buzzer_alert()
    post_alert(label, confidence, frame_bgr=frame)
else:
    print(f"No alert - '{label}' does not meet the distraction-alert criteria "
          f"(either it's safe_driving, or confidence {confidence:.0%} is below the "
          f"{CONFIDENCE_THRESHOLD:.0%} threshold).")